In [ ]:

# Cell 2: Load and Explore Data
# Diabetes dataset for regression
X, y = load_diabetes(return_X_y=True)
feature_names = load_diabetes().feature_names
print(f"Dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Features: {feature_names}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")




In [ ]:
# Cell 2: Load and Explore Data
# Diabetes dataset for regression
X, y = load_diabetes(return_X_y=True)
feature_names = load_diabetes().feature_names
print(f"Dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Features: {feature_names}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
# Cell 3: Train Model
model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Baseline performance
y_pred = model.predict(X_test)
baseline_r2 = r2_score(y_test, y_pred)
baseline_mse = mean_squared_error(y_test, y_pred)

print(f"Baseline R² Score: {baseline_r2:.4f}")
print(f"Baseline MSE: {baseline_mse:.4f}")

In [ ]:
# Cell 4: Implement Permutation Importance From Scratch
def permutation_importance_manual(model, X, y, n_repeats=10, random_state=42):
    """
    Calculate permutation feature importance.
    
    Parameters:
    -----------
    model: Fitted model with .predict() method
    X: Feature matrix (n_samples, n_features)
    y: Target vector
    n_repeats: Number of permutation repeats
    random_state: Seed for reproducibility
    
    Returns:
    --------
    importance_mean: Mean importance for each feature
    importance_std: Standard deviation across repeats
    """
    baseline = r2_score(y, model.predict(X))
    n_features = X.shape[1]
    importances = np.zeros((n_features, n_repeats))
    
    for feature_idx in range(n_features):
        for repeat in range(n_repeats):
            # Create copy and shuffle feature
            X_permuted = X.copy()
            rng = np.random.RandomState(random_state + repeat)
            rng.shuffle(X_permuted[:, feature_idx])
            
            # Get permuted score
            permuted_score = r2_score(y, model.predict(X_permuted))
            
            # Calculate importance
            importance = baseline - permuted_score
            importances[feature_idx, repeat] = importance
    
    importance_mean = np.mean(importances, axis=1)
    importance_std = np.std(importances, axis=1)
    
    return importance_mean, importance_std, importances

# Calculate PFI
print("Calculating Permutation Feature Importance...")
importance_mean, importance_std, importances_all = permutation_importance_manual(
    model, X_test, y_test, n_repeats=20, random_state=42
)
print("✓ Done!")


In [ ]:
# Cell 5: Display Results
results_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_mean,
    'Std Dev': importance_std,
    'Lower CI': importance_mean - 2 * importance_std,
    'Upper CI': importance_mean + 2 * importance_std
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nPermutation Feature Importance Results:")
print(results_df.to_string())

In [ ]:
# Cell 6: Visualize Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Horizontal bar chart with error bars
sorted_idx = np.argsort(importance_mean)[::-1]
ax1.barh(range(len(sorted_idx)), 
         importance_mean[sorted_idx],
         xerr=importance_std[sorted_idx],
         alpha=0.8,
         color='steelblue',
         error_kw={'elinewidth': 2, 'capsize': 5})
ax1.set_yticks(range(len(sorted_idx)))
ax1.set_yticklabels([feature_names[i] for i in sorted_idx])
ax1.set_xlabel('Importance (R² Drop)', fontweight='bold')
ax1.set_title('Permutation Feature Importance', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Distribution of importance values
positions = np.arange(len(feature_names))
bp = ax2.boxplot([importances_all[i] for i in sorted_idx],
                   labels=[feature_names[i] for i in sorted_idx],
                   vert=False,
                   patch_artist=True)

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax2.set_xlabel('Importance Value', fontweight='bold')
ax2.set_title('Importance Distribution (20 repeats)', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Comparison with Built-in Method
from sklearn.inspection import permutation_importance as sklearn_pfi

result_sklearn = sklearn_pfi(model, X_test, y_test, n_repeats=20, random_state=42)

comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'Our Implementation': importance_mean,
    'Scikit-learn': result_sklearn.importances_mean,
    'Difference': np.abs(importance_mean - result_sklearn.importances_mean)
}).sort_values('Our Implementation', ascending=False)

print("\nComparison with Scikit-learn Implementation:")
print(comparison_df.to_string())


In [ ]:
# Cell 8: Analysis - Feature Correlation
print("\n" + "="*60)
print("ANALYSIS: Understanding the Results")
print("="*60)

# Check correlations
X_test_df = pd.DataFrame(X_test, columns=feature_names)
correlation_with_target = pd.Series({
    name: np.corrcoef(X_test_df[name], y_test)[0, 1]
    for name in feature_names
})

analysis_df = pd.DataFrame({
    'Feature': feature_names,
    'PFI Importance': importance_mean,
    'Correlation with Target': correlation_with_target.values
}).sort_values('PFI Importance', ascending=False)

print("\nFeature Analysis:")
print(analysis_df.to_string())

print("\n💡 Key Insights:")
print(f"- Most important feature: {analysis_df.iloc[0]['Feature']} (importance: {analysis_df.iloc[0]['PFI Importance']:.6f})")
print(f"- Least important feature: {analysis_df.iloc[-1]['Feature']} (importance: {analysis_df.iloc[-1]['PFI Importance']:.6f})")


In [ ]:
# Cell 9: Limitations and Considerations
print("\n" + "="*60)
print("LIMITATIONS OF PERMUTATION FEATURE IMPORTANCE")
print("="*60)

# Simulate correlated features problem
print("\n1. CORRELATED FEATURES PROBLEM:")
print("-" * 40)

# Create synthetic correlated features
X_test_synthetic = X_test.copy()
X_test_synthetic[:, 1] = X_test_synthetic[:, 0] + np.random.normal(0, 0.1, len(X_test))

pfi_synthetic, _, _ = permutation_importance_manual(model, X_test_synthetic, y_test, n_repeats=10)

print("With correlated features, both may show low importance")
print(f"Feature 0 importance: {pfi_synthetic[0]:.6f}")
print(f"Feature 1 importance (correlated): {pfi_synthetic[1]:.6f}")
print("→ Both features still convey information despite correlation")

In [ ]:
# Cell 10: Summary and Recommendations
print("\n" + "="*60)
print("SUMMARY & BEST PRACTICES")
print("="*60)

summary = """
✓ STRENGTHS:
  • Model-agnostic: works with any model type
  • Easy to understand and implement
  • Shows practical importance (what impacts predictions)
  • Provides uncertainty estimates (std dev)

✗ LIMITATIONS:
  • Struggles with highly correlated features
  • Computationally expensive for large datasets
  • Permutations may create unrealistic data combinations
  • Results depend on test set selection

⚠️ BEST PRACTICES:
  1. Use multiple repeats (10-20) for stability
  2. Normalize importance values if comparing across models
  3. Always visualize with error bars
  4. Check feature correlations separately
  5. Combine with other interpretation methods (SHAP, LIME)

📚 FURTHER READING:
  • "Interpretable Machine Learning" by Christoph Molnar
  • "The Foundations of Machine Learning" - Chapter on Explainability
  • Scikit-learn documentation on permutation importance
"""

print(summary)
